# Section 05: Maintaining and Automating Data Workloads

**CareerAlign GCP Professional Data Engineer**

This notebook covers:
1. BigQuery cost analysis with INFORMATION_SCHEMA
2. BigQuery Editions, reservations, and slot management
3. Cloud Composer DAG patterns (task groups, branching, error handling)
4. Dataflow pipeline monitoring and optimization
5. Backup and recovery (time travel, snapshots)
6. Monitoring and alerting patterns

> **Note**: Cells marked with `# REQUIRES GCP` need a GCP project. Local-only cells run anywhere.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-pde/notebooks/05-maintaining-data-workloads.ipynb)

---
## Setup & Installations

In [ ]:
!pip install -q google-cloud-bigquery google-cloud-monitoring pandas numpy

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

---
## 1. BigQuery Cost Analysis

Use INFORMATION_SCHEMA to analyze query costs and identify optimization opportunities.

In [ ]:
# --- Simulate INFORMATION_SCHEMA.JOBS data ---

np.random.seed(42)
n_queries = 200

users = np.random.choice([
    'analyst1@corp.com', 'analyst2@corp.com', 'etl-svc@corp.iam',
    'dashboards-svc@corp.iam', 'adhoc-user@corp.com', 'data-eng@corp.com'
], n_queries, p=[0.25, 0.2, 0.25, 0.15, 0.1, 0.05])

bytes_billed = np.random.lognormal(28, 2, n_queries).astype(int)  # Varies widely
slot_ms = np.random.lognormal(12, 1.5, n_queries).astype(int)
durations = np.random.lognormal(2, 1, n_queries).round(1)

jobs_df = pd.DataFrame({
    'user_email': users,
    'total_bytes_billed': bytes_billed,
    'total_slot_ms': slot_ms,
    'duration_sec': durations,
    'creation_time': pd.date_range('2025-02-01', periods=n_queries, freq='2h'),
})

# Analyze like INFORMATION_SCHEMA query
cost_per_tb = 6.25
user_costs = jobs_df.groupby('user_email').agg(
    query_count=('total_bytes_billed', 'count'),
    total_tb_billed=('total_bytes_billed', lambda x: round(x.sum() / 1e12, 4)),
    avg_slot_sec=('total_slot_ms', lambda x: round(x.mean() / 1000, 1)),
    avg_duration=('duration_sec', 'mean'),
).round(2)
user_costs['est_cost_usd'] = (user_costs['total_tb_billed'] * cost_per_tb).round(2)
user_costs = user_costs.sort_values('est_cost_usd', ascending=False)

print("=== BigQuery Cost Analysis (simulated INFORMATION_SCHEMA.JOBS) ===")
print(f"Period: 30 days, {n_queries} queries")
print()
print(user_costs.to_string())
print(f"\nTotal estimated cost: ${user_costs['est_cost_usd'].sum():.2f}")

In [ ]:
# --- INFORMATION_SCHEMA SQL queries for cost analysis ---

cost_queries = """
-- Top 20 most expensive queries in last 30 days
SELECT
  job_id,
  user_email,
  ROUND(total_bytes_billed / POW(1024, 4), 4) AS tb_billed,
  ROUND(total_bytes_billed / POW(1024, 4) * 6.25, 2) AS cost_usd,
  total_slot_ms,
  TIMESTAMP_DIFF(end_time, start_time, SECOND) AS duration_sec,
  SUBSTR(query, 1, 100) AS query_preview
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
  AND job_type = 'QUERY' AND state = 'DONE'
ORDER BY total_bytes_billed DESC
LIMIT 20;

-- Cost per user per day
SELECT
  DATE(creation_time) AS query_date,
  user_email,
  COUNT(*) AS queries,
  ROUND(SUM(total_bytes_billed) / POW(1024, 4) * 6.25, 2) AS daily_cost
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND job_type = 'QUERY'
GROUP BY 1, 2
ORDER BY daily_cost DESC;

-- Identify queries that don't use partition pruning
SELECT
  job_id,
  user_email,
  total_bytes_billed,
  SUBSTR(query, 1, 200) AS query_preview
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND total_bytes_billed > 10 * POW(1024, 3)  -- > 10 GB
  AND query NOT LIKE '%WHERE%partition_column%'
ORDER BY total_bytes_billed DESC;

-- Set maximum bytes billed per query (safety guard)
-- Add to query: OPTIONS (maximum_bytes_billed = 10737418240)  -- 10 GB
"""

print("=== Cost Analysis SQL Queries ===")
print(cost_queries)

---
## 2. BigQuery Editions and Reservations

Understand slot-based pricing and workload isolation.

In [ ]:
# --- Editions Pricing Comparison ---

editions = pd.DataFrame({
    'Edition': ['Standard', 'Enterprise', 'Enterprise Plus'],
    '$/slot-hr (PAYG)': [0.04, 0.06, 0.10],
    '$/slot-hr (1yr)': [0.032, 0.048, 0.080],
    '$/slot-hr (3yr)': [0.024, 0.036, 0.060],
    'Key Features': [
        'Baseline + autoscaling, basic',
        'CMEK, multi-region, BI Engine, mat views',
        'Advanced security, cross-region failover, highest SLA'
    ]
})

print("=== BigQuery Editions Pricing ===")
print(editions.to_string(index=False))
print()

# Cost comparison: On-demand vs Editions
monthly_tb = 50  # 50 TB scanned per month
on_demand = monthly_tb * 6.25

slots_needed = 200  # Estimated slots needed
hours_per_month = 730
editions_cost = slots_needed * 0.04 * hours_per_month  # Standard PAYG
editions_1yr = slots_needed * 0.032 * hours_per_month   # Standard 1-yr

print("=== Cost Comparison: 50 TB/month scanned ===")
print(f"  On-demand:            ${on_demand:,.0f}/month")
print(f"  Standard PAYG (200s): ${editions_cost:,.0f}/month")
print(f"  Standard 1-yr (200s): ${editions_1yr:,.0f}/month")
print()
if editions_cost < on_demand:
    print(f"  Editions saves ${on_demand - editions_cost:,.0f}/month ({(1-editions_cost/on_demand)*100:.0f}%)")
else:
    print(f"  On-demand saves ${editions_cost - on_demand:,.0f}/month")

In [ ]:
# --- Reservation Setup Commands ---

reservation_cmds = """
# Step 1: Create capacity commitment (500 slots baseline)
bq mk --capacity_commitment \\
  --project_id=admin-project \\
  --location=US \\
  --slots=500 \\
  --plan=ANNUAL \\
  --edition=ENTERPRISE

# Step 2: Create reservations (split slots for workload isolation)
bq mk --reservation \\
  --project_id=admin-project --location=US \\
  --reservation_id=etl_jobs \\
  --slots=200 --autoscale_max_slots=400

bq mk --reservation \\
  --project_id=admin-project --location=US \\
  --reservation_id=analytics \\
  --slots=200 --autoscale_max_slots=500

bq mk --reservation \\
  --project_id=admin-project --location=US \\
  --reservation_id=adhoc \\
  --slots=100 --autoscale_max_slots=200

# Step 3: Assign projects to reservations
bq mk --reservation_assignment \\
  --project_id=admin-project --location=US \\
  --reservation_id=etl_jobs \\
  --assignee_id=etl-project \\
  --assignee_type=PROJECT --job_type=QUERY

bq mk --reservation_assignment \\
  --project_id=admin-project --location=US \\
  --reservation_id=analytics \\
  --assignee_id=analytics-project \\
  --assignee_type=PROJECT --job_type=QUERY
"""

print("=== BigQuery Reservation Setup ===")
print(reservation_cmds)
print("Key concepts:")
print("  - Baseline slots: guaranteed capacity")
print("  - Autoscale max: burst capacity (billed per second)")
print("  - Idle slots: automatically shared across reservations")
print("  - Workload isolation: ETL can't starve analytics queries")

---
## 3. Cloud Composer DAG Patterns

Advanced DAG patterns: task groups, branching, error handling.

In [ ]:
# --- Advanced DAG Pattern: Branching + Error Handling ---

dag_pattern = '''
from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.operators.empty import EmptyOperator
from airflow.utils.task_group import TaskGroup
from airflow.utils.trigger_rule import TriggerRule
from datetime import timedelta

default_args = {
    "owner": "data-eng",
    "retries": 3,
    "retry_delay": timedelta(minutes=5),
    "retry_exponential_backoff": True,
    "max_retry_delay": timedelta(minutes=60),
    "email_on_failure": True,
    "email": ["data-eng-alerts@corp.com"],
    "sla": timedelta(hours=2),
    "execution_timeout": timedelta(hours=4),
}

def check_data_quality(**context):
    """Check quality and decide routing."""
    score = context["ti"].xcom_pull(task_ids="run_quality_checks")
    if score >= 0.95:
        return "proceed_production"
    elif score >= 0.80:
        return "proceed_with_warning"
    else:
        return "quarantine_data"

with DAG(
    dag_id="production_pipeline_v2",
    default_args=default_args,
    schedule_interval="0 6 * * *",
    catchup=False,
    tags=["production"],
) as dag:

    # Ingest phase (parallel tasks)
    with TaskGroup("ingest") as ingest:
        ingest_orders = PythonOperator(task_id="orders", ...)
        ingest_users = PythonOperator(task_id="users", ...)

    # Quality check + branching
    quality = PythonOperator(task_id="run_quality_checks", ...)
    branch = BranchPythonOperator(
        task_id="quality_branch",
        python_callable=check_data_quality,
    )

    # Three possible paths
    proceed_prod = PythonOperator(task_id="proceed_production", ...)
    proceed_warn = PythonOperator(task_id="proceed_with_warning", ...)
    quarantine = PythonOperator(task_id="quarantine_data", ...)

    # Notification (runs regardless of which branch was taken)
    notify = PythonOperator(
        task_id="send_notification",
        trigger_rule=TriggerRule.ONE_SUCCESS,  # Run if any upstream succeeded
        ...
    )

    ingest >> quality >> branch
    branch >> [proceed_prod, proceed_warn, quarantine]
    [proceed_prod, proceed_warn, quarantine] >> notify
'''

print("=== Advanced Composer DAG Pattern ===")
print(dag_pattern)

In [ ]:
# --- Automation Decision Matrix ---

automation = pd.DataFrame({
    'Need': [
        'Simple scheduled SQL',
        'SQL transformations with dependencies',
        'Multi-service pipeline orchestration',
        'Simple API workflow',
        'Event-triggered processing',
        'Cron-triggered HTTP call',
    ],
    'Tool': [
        'BigQuery Scheduled Queries',
        'Dataform',
        'Cloud Composer',
        'Workflows',
        'Cloud Functions + Eventarc',
        'Cloud Scheduler',
    ],
    'Complexity': ['Low', 'Medium', 'High', 'Low', 'Medium', 'Low'],
    'Cost': ['Free (query cost only)', 'Free (BQ cost)', 'Medium (Airflow env)',
             'Very low (per step)', 'Low (per invocation)', 'Very low (per job)'],
})

print("=== Automation Tool Selection ===")
print(automation.to_string(index=False))

---
## 4. Dataflow Monitoring and Optimization

In [ ]:
# --- Dataflow Key Metrics ---

dataflow_metrics = pd.DataFrame({
    'Metric': [
        'system_lag',
        'data_watermark_age',
        'elements_count',
        'cpu_utilization',
        'memory_utilization',
        'backlog_bytes',
        'current_num_workers',
    ],
    'Description': [
        'Time between newest unprocessed element and now',
        'Age of the oldest unprocessed element',
        'Total elements processed',
        'Worker CPU usage percentage',
        'Worker memory usage percentage',
        'Bytes of unprocessed data in source',
        'Number of active workers',
    ],
    'Alert Threshold': [
        '> 5 minutes (streaming)',
        '> 10 minutes',
        'Sudden drop = problem',
        '> 80% sustained',
        '> 85%',
        'Growing continuously',
        'At max (hitting autoscale limit)',
    ],
    'Action': [
        'Increase workers, optimize DoFns',
        'Check source connectivity, worker health',
        'Check for pipeline errors',
        'Scale up workers or optimize code',
        'Increase worker memory or reduce batch size',
        'Scale up or fix bottleneck',
        'Increase max_num_workers',
    ]
})

print("=== Dataflow Key Metrics ===")
for _, row in dataflow_metrics.iterrows():
    print(f"\n  {row['Metric']}:")
    print(f"    What: {row['Description']}")
    print(f"    Alert: {row['Alert Threshold']}")
    print(f"    Fix: {row['Action']}")

In [ ]:
# --- Dataflow Optimization Tips ---

optimization_tips = """
=== Dataflow Optimization Checklist ===

1. Use Streaming Engine (--enable_streaming_engine)
   - Offloads shuffle to managed service
   - Reduces worker CPU and memory

2. Use Combine instead of GroupByKey + manual aggregation
   - Combine does partial aggregation before shuffle
   - Dramatically reduces data shuffled

3. Use side inputs for small lookup data
   - Avoid CoGroupByKey for enrichment
   - Side inputs broadcast to all workers

4. Avoid hot keys (data skew)
   - One key with 90% of data = one worker bottleneck
   - Use Reshuffle or composite keys to distribute

5. Right-size workers
   - CPU-bound: use c2/c3 machine types
   - Memory-bound: use n2-highmem
   - Default n1-standard-4 is often fine

6. Dataflow Prime (auto-optimization)
   - Automatically selects worker types
   - Right-sizes memory and CPU per stage
   - Enable with --dataflow_service_options=enable_prime

7. Drain vs Cancel
   - Drain: gracefully finish in-flight, then stop
   - Cancel: immediately stop (may lose in-flight data)
   - Always use drain for production shutdowns

8. Update streaming pipelines
   - --update flag preserves pipeline state
   - --transform_name_mapping for renamed transforms
   - No data loss during update
"""

print(optimization_tips)

---
## 5. Backup and Recovery

In [ ]:
# --- BigQuery Time Travel and Snapshots ---

backup_sql = """
-- Time travel: query data as it was N hours ago
SELECT * FROM `project.dataset.events`
FOR SYSTEM_TIME AS OF 
  TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 2 HOUR)
WHERE DATE(event_time) = '2025-03-01';

-- Restore deleted/modified rows from time travel
INSERT INTO `project.dataset.events`
SELECT * FROM `project.dataset.events`
FOR SYSTEM_TIME AS OF TIMESTAMP('2025-03-01 00:00:00 UTC')
WHERE order_id IN ('deleted_order_1', 'deleted_order_2');

-- Create a table snapshot (point-in-time, space-efficient)
CREATE SNAPSHOT TABLE `project.dataset.events_snapshot_mar01`
CLONE `project.dataset.events`
FOR SYSTEM_TIME AS OF '2025-03-01 00:00:00 UTC'
OPTIONS (expiration_timestamp = 
  TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 30 DAY));

-- Restore from snapshot
CREATE OR REPLACE TABLE `project.dataset.events`
CLONE `project.dataset.events_snapshot_mar01`;

-- Configure time travel window (default 7 days, min 2 days)
ALTER TABLE `project.dataset.events`
SET OPTIONS (max_time_travel_hours = 168);  -- 7 days
"""

print("=== BigQuery Time Travel & Snapshots ===")
print(backup_sql)

In [ ]:
# --- Backup strategies per service ---

backup_strategies = pd.DataFrame({
    'Service': ['BigQuery', 'Cloud SQL', 'Cloud Spanner', 'Bigtable',
                'Cloud Storage', 'AlloyDB', 'Firestore'],
    'Backup Method': [
        'Time travel (7d), snapshots, dataset copies',
        'Auto daily + on-demand + PITR (binlog/WAL)',
        'Auto backups, export to GCS',
        'Managed backups (up to 30 days)',
        'Object versioning, cross-region replication',
        'Continuous backup + PITR (14 days)',
        'Scheduled exports to GCS, PITR'
    ],
    'Recovery Time': [
        'Instant (time travel query)',
        'Minutes (restore from backup)',
        'Minutes (restore from backup)',
        'Minutes to hours (depends on size)',
        'Instant (restore version)',
        'Minutes (PITR)',
        'Minutes (import from GCS)'
    ],
    'HA/Failover': [
        'Built-in (multi-zone). Cross-region with Enterprise+.',
        'Regional HA (zone failover). Cross-region read replicas.',
        'Multi-region configs (99.999% SLA).',
        'Multi-cluster replication.',
        'Dual/multi-region buckets.',
        'Regional HA with auto-failover.',
        'Multi-region (99.999% SLA).'
    ]
})

print("=== Backup & Recovery per Service ===")
for _, row in backup_strategies.iterrows():
    print(f"\n  {row['Service']}:")
    print(f"    Backup:   {row['Backup Method']}")
    print(f"    Recovery: {row['Recovery Time']}")
    print(f"    HA:       {row['HA/Failover']}")

---
## 6. Monitoring and Alerting

In [ ]:
# --- Cloud Monitoring alert examples ---

monitoring_cmds = """
# Alert: Pub/Sub subscription backlog growing
gcloud alpha monitoring policies create \\
  --display-name="PubSub High Backlog" \\
  --condition-display-name="Unacked messages > 10K" \\
  --condition-filter='resource.type="pubsub_subscription" 
    AND metric.type="pubsub.googleapis.com/subscription/num_undelivered_messages"' \\
  --condition-threshold-value=10000 \\
  --condition-threshold-duration=300s \\
  --notification-channels=CHANNEL_ID

# Alert: Dataflow system lag > 5 minutes
# Metric: dataflow.googleapis.com/job/system_lag
# Threshold: > 300 seconds for 5 minutes

# Alert: BigQuery slot utilization > 90%
# Metric: bigquery.googleapis.com/slots/total_available
# vs bigquery.googleapis.com/slots/allocated_for_project

# Alert: Cloud Composer DAG failure
# Metric: composer.googleapis.com/environment/dagrun/count
# Filter: state="failed"
# Threshold: > 0 in any 15-minute window

# Useful Cloud Logging queries:
# BigQuery slow queries:
#   resource.type="bigquery_resource" 
#   protoPayload.serviceData.queryResponse.totalBilledBytes > 10000000000

# Dataflow errors:
#   resource.type="dataflow_step"
#   severity>=ERROR

# Composer task failures:
#   resource.type="cloud_composer_environment"
#   "Task failed" OR "task_failed_dep"
"""

print("=== Cloud Monitoring & Alerting ===")
print(monitoring_cmds)

In [ ]:
# --- Key monitoring metrics per service ---

metrics = pd.DataFrame({
    'Service': ['BigQuery', 'BigQuery', 'Dataflow', 'Dataflow',
                'Pub/Sub', 'Pub/Sub', 'Composer', 'Composer',
                'Bigtable', 'Cloud SQL'],
    'Metric': [
        'Slot utilization', 'Bytes scanned per query',
        'System lag', 'Worker CPU utilization',
        'Oldest unacked message age', 'Subscription backlog (bytes)',
        'DAG run duration', 'Task failure count',
        'Server error count', 'CPU utilization'
    ],
    'What to Watch': [
        'Near 100% = need more slots',
        'Sudden spikes = missing partition filter',
        'Growing = pipeline falling behind',
        'Sustained > 80% = scale up',
        'Growing = consumers too slow',
        'Increasing = consumer not keeping up',
        'Exceeding SLA = investigate bottleneck',
        'Any failures = check logs',
        'Spikes = check row key design (hotspot)',
        'Sustained > 80% = scale up instance'
    ]
})

print("=== Key Monitoring Metrics ===")
print(metrics.to_string(index=False))

---
## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **Cost Analysis** | INFORMATION_SCHEMA.JOBS for query costs. Track bytes billed, slot usage per user. |
| **Editions** | Standard/Enterprise/Enterprise Plus. Baseline + autoscale. 1yr/3yr commitments for discounts. |
| **Reservations** | Workload isolation: ETL, analytics, ad-hoc. Idle slots auto-shared. |
| **Composer DAGs** | Task groups, branching, trigger rules, SLA monitoring, exponential backoff. |
| **Dataflow Ops** | Monitor system_lag, use Streaming Engine, Drain (not Cancel) for graceful shutdown. |
| **Backup** | BQ time travel (7d), snapshots. Cloud SQL PITR. Spanner multi-region. |
| **Monitoring** | Cloud Monitoring alerts for backlog, lag, slot usage. Cloud Logging for debugging. |